# Lightweight ViT Multi-Task Learning: Inference Demo

This notebook demonstrates the multi-task capabilities (Object Classification + Attribute Prediction) of the trained models. It also includes a basic Text-to-Image retrieval system.

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

from src.models.multitask_vit import MultiTaskViT

# Configuration
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE = 224

preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [ ]:
def load_model(ckpt_path, backbone_name='deit_tiny_patch16_224'):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    classes = ckpt['classes']
    attr_maps = ckpt['attr_value2idx']
    attr_sizes = {a: len(v) for a, v in attr_maps.items()}
    
    model = MultiTaskViT(backbone_name, len(classes), attr_sizes, pretrained=False)
    model.load_state_dict(ckpt['model_state'])
    model.to(DEVICE).eval()
    return model, classes, attr_maps

## Perform Inference

In [ ]:
def predict(model, classes, attr_maps, img_path):
    img = Image.open(img_path).convert('RGB')
    x = preprocess(img).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        cls_logits, attr_logits, _ = model(x)
        
        # Object Class
        cls_prob = F.softmax(cls_logits, dim=1)[0]
        pred_cls = classes[cls_prob.argmax()]
        
        # Attributes
        results = {
            'class': pred_cls,
            'attributes': {}
        }
        
        for a, logits in attr_logits.items():
            attr_prob = F.softmax(logits, dim=1)[0]
            idx = attr_prob.argmax().item()
            inv_map = {v: k for k, v in attr_maps[a].items()}
            results['attributes'][a] = inv_map[idx]
            
    return results, img